#  Comprehensive GAN Evaluation Metrics

This notebook evaluates trained GAN generators using multiple metrics:

1. **FID (Fréchet Inception Distance)** - Measures distribution similarity (lower is better, <30 is excellent)
2. **IS (Inception Score)** - Measures quality and diversity (higher is better, >8 is good for digits)
3. **Precision** - Fraction of generated images that look realistic (0-1, higher is better)
4. **Recall** - Coverage of real data manifold (0-1, higher is better)

**Goal:** Evaluate your trained generator and diagnose any quality issues.

In [1]:
# Import libraries
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import pandas as pd
from scipy.linalg import sqrtm
from torchvision.models import inception_v3
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import pairwise_distances
import json

# Add project root to path
project_root = Path.cwd()
src_root = project_root / "src"
sys.path.insert(0, str(src_root))

from nepscript.models.factory import create_models_from_config

print(" All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

✅ All libraries imported successfully!
PyTorch version: 2.9.0+cpu
CUDA available: False


## 1️ Configuration: Specify Model to Evaluate

Change these paths to point to your trained generator model.

In [8]:
# Configuration
MODEL_PATH = "experiments/gan_run_models_and_images/final_training/progressive-phase-2/models/generator_epoch_0250.pth"
CONFIG_PATH = "experiments/gan_run_models_and_images/nas_results/best_architecture_progressive.json"

# Data paths
DATA_DIR = "data/DevanagariHandwrittenDigitDataset"
LABELS_CSV = "data/hindi_mnist.csv"

# Evaluation settings
NUM_SAMPLES = 1  # Number of images to generate for evaluation
BATCH_SIZE = 64
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f" Model: {MODEL_PATH}")
print(f" Config: {CONFIG_PATH}")
print(f" Device: {DEVICE}")
print(f" Samples: {NUM_SAMPLES}")

 Model: experiments/gan_run_models_and_images/final_training/progressive-phase-2/models/generator_epoch_0250.pth
 Config: experiments/gan_run_models_and_images/nas_results/best_architecture_progressive.json
 Device: cpu
 Samples: 1


## 2️ Load Pre-trained Inception Model

We use Inception V3 to extract features for FID, IS, Precision, and Recall calculations.

In [9]:
# Load Inception V3 model
print("Loading Inception V3 model...")
inception_model = inception_v3(pretrained=True, transform_input=False)
inception_model.fc = nn.Identity()  # Remove final classification layer
inception_model = inception_model.to(DEVICE)
inception_model.eval()

print("Inception V3 loaded successfully!")

Loading Inception V3 model...


C:\Users\HP\.conda\envs\nepscript\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\HP\.conda\envs\nepscript\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Inception V3 loaded successfully!


## 3️ Load Real Images Dataset

In [10]:
# Dataset class for real images
class RealDataset(Dataset):
    def __init__(self, csv_file, root_dir, split=None, transform=None):
        self.data = pd.read_csv(csv_file)
        if split is not None:
            self.data = self.data[self.data['filename'].str.contains(split)]
        self.root_dir = root_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_path = Path(self.root_dir) / self.data.iloc[idx]['filename']
        image = Image.open(img_path).convert('RGB')  # Convert to RGB for Inception
        if self.transform:
            image = self.transform(image)
        return image

# Transform for Inception (needs 299x299 RGB images)
transform_inception = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load real dataset
real_dataset = RealDataset(LABELS_CSV, DATA_DIR, split=None, transform=transform_inception)
real_loader = DataLoader(real_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

print(f" Real dataset loaded: {len(real_dataset):,} images")

 Real dataset loaded: 20,000 images


## 4️ Load Trained Generator and Generate Fake Images

In [11]:
# Load generator configuration and model
with open(CONFIG_PATH, 'r') as f:
    config = json.load(f)

print(f"Creating generator from config...")
generator, _ = create_models_from_config(config, DEVICE)
generator.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
generator.eval()

latent_dim = config['generator']['latent_dim']
print(f" Generator loaded! Latent dim: {latent_dim}")

# Generate fake images
print(f"\nGenerating {NUM_SAMPLES} fake images...")
fake_images = []

with torch.no_grad():
    num_batches = (NUM_SAMPLES + BATCH_SIZE - 1) // BATCH_SIZE
    
    for i in range(num_batches):
        current_batch_size = min(BATCH_SIZE, NUM_SAMPLES - i * BATCH_SIZE)
        
        # Generate random latent vectors and labels
        z = torch.randn(current_batch_size, latent_dim).to(DEVICE)
        labels = torch.randint(0, 10, (current_batch_size,)).to(DEVICE)
        
        # Generate images
        generated = generator(z, labels)
        
        # Convert from [-1, 1] to [0, 1] and prepare for Inception
        generated = (generated + 1) / 2  # [-1, 1] -> [0, 1]
        generated = generated.repeat(1, 3, 1, 1)  # Grayscale -> RGB
        
        # Resize to 299x299 for Inception
        generated_resized = F.interpolate(generated, size=(299, 299), mode='bilinear', align_corners=False)
        
        # Normalize for Inception
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(DEVICE)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(DEVICE)
        generated_normalized = (generated_resized - mean) / std
        
        fake_images.append(generated_normalized.cpu())

fake_images = torch.cat(fake_images, dim=0)
print(f" Generated {len(fake_images):,} fake images")

Creating generator from config...
 Generator loaded! Latent dim: 64

Generating 1 fake images...
 Generator loaded! Latent dim: 64

Generating 1 fake images...
 Generated 1 fake images
 Generated 1 fake images


## 5️ Extract Features from Real and Fake Images

In [12]:
# Extract features from real images
print("Extracting features from real images...")
real_features = []
real_probabilities = []

with torch.no_grad():
    for batch in real_loader:
        batch = batch.to(DEVICE)
        features = inception_model(batch)
        real_features.append(features.cpu().numpy())
        
        if len(np.concatenate(real_features)) >= NUM_SAMPLES:
            break

real_features = np.concatenate(real_features)[:NUM_SAMPLES]
print(f" Real features extracted: {real_features.shape}")

# Extract features from fake images
print("Extracting features from fake images...")
fake_features = []

with torch.no_grad():
    for i in range(0, len(fake_images), BATCH_SIZE):
        batch = fake_images[i:i+BATCH_SIZE].to(DEVICE)
        features = inception_model(batch)
        fake_features.append(features.cpu().numpy())

fake_features = np.concatenate(fake_features)
print(f" Fake features extracted: {fake_features.shape}")

Extracting features from real images...
 Real features extracted: (1, 2048)
Extracting features from fake images...
 Real features extracted: (1, 2048)
Extracting features from fake images...
 Fake features extracted: (1, 2048)
 Fake features extracted: (1, 2048)


## 6️ Calculate FID (Fréchet Inception Distance)

FID measures the distance between real and generated image distributions. Lower is better.
- **< 10**: Excellent
- **10-30**: Good  
- **30-50**: Acceptable
- **> 100**: Poor (mode collapse or training failure)

In [ ]:
# Calculate FID
def calculate_fid(real_features, fake_features):
    # Calculate mean and covariance
    mu_real = np.mean(real_features, axis=0)
    sigma_real = np.cov(real_features, rowvar=False)
    
    mu_fake = np.mean(fake_features, axis=0)
    sigma_fake = np.cov(fake_features, rowvar=False)
    
    # Calculate FID
    diff = mu_real - mu_fake
    covmean = sqrtm(sigma_real.dot(sigma_fake))
    
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    
    fid = diff.dot(diff) + np.trace(sigma_real + sigma_fake - 2 * covmean)
    return fid

fid_score = calculate_fid(real_features, fake_features)

print(f"\n{'='*60}")
print(f" FID SCORE: {fid_score:.2f}")
print(f"{'='*60}")

if fid_score < 10:
    verdict = " EXCELLENT - Nearly perfect distribution match!"
elif fid_score < 30:
    verdict = " GOOD - Quality generation with minor differences"
elif fid_score < 50:
    verdict = "  ACCEPTABLE - Noticeable but reasonable differences"
elif fid_score < 100:
    verdict = " POOR - Significant distribution mismatch"
else:
    verdict = " VERY POOR - Mode collapse or severe training issues"

print(f"\n{verdict}")

C:\Users\HP\AppData\Local\Temp\ipykernel_4556\2842888015.py:5: RuntimeWarning: Degrees of freedom <= 0 for slice
  sigma_real = np.cov(real_features, rowvar=False)
C:\Users\HP\.conda\envs\nepscript\Lib\site-packages\numpy\lib\_function_base_impl.py:2894: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
C:\Users\HP\.conda\envs\nepscript\Lib\site-packages\numpy\lib\_function_base_impl.py:2894: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
C:\Users\HP\AppData\Local\Temp\ipykernel_4556\2842888015.py:8: RuntimeWarning: Degrees of freedom <= 0 for slice
  sigma_fake = np.cov(fake_features, rowvar=False)


## 7️ Calculate Inception Score (IS)

IS measures quality and diversity. Higher is better.
- **> 8**: Excellent for digits
- **6-8**: Good
- **3-6**: Acceptable
- **< 3**: Poor quality or low diversity

In [ ]:
# Get class probabilities from Inception (with classifier)
inception_classifier = inception_v3(pretrained=True, transform_input=False).to(DEVICE)
inception_classifier.eval()

print("Getting class probabilities for Inception Score...")
fake_probabilities = []

with torch.no_grad():
    for i in range(0, len(fake_images), BATCH_SIZE):
        batch = fake_images[i:i+BATCH_SIZE].to(DEVICE)
        logits = inception_classifier(batch)
        probs = F.softmax(logits, dim=1)
        fake_probabilities.append(probs.cpu().numpy())

fake_probabilities = np.concatenate(fake_probabilities)

# Calculate Inception Score
def calculate_inception_score(probs, splits=10):
    scores = []
    
    for i in range(splits):
        part = probs[i * (len(probs) // splits): (i + 1) * (len(probs) // splits)]
        kl_divs = part * (np.log(part) - np.log(np.expand_dims(np.mean(part, axis=0), 0)))
        kl_div_mean = np.mean(np.sum(kl_divs, axis=1))
        scores.append(np.exp(kl_div_mean))
    
    return np.mean(scores), np.std(scores)

is_score, is_std = calculate_inception_score(fake_probabilities)

print(f"\n{'='*60}")
print(f" INCEPTION SCORE: {is_score:.2f} ± {is_std:.2f}")
print(f"{'='*60}")

if is_score > 8:
    verdict = " EXCELLENT - High quality and diversity!"
elif is_score > 6:
    verdict = " GOOD - Solid quality and variety"
elif is_score > 3:
    verdict = "  ACCEPTABLE - Room for improvement"
else:
    verdict = " POOR - Low quality or diversity issues"

print(f"\n{verdict}")

## 8️ Calculate Precision and Recall

- **Precision**: Fraction of generated images that look realistic (how many fakes fool us?)
- **Recall**: Coverage of real data (does generator cover all digit variations?)

Both should be > 0.7 for good performance.

In [ ]:
# Calculate Precision and Recall
def calculate_precision_recall(real_features, fake_features, k=3):
    print("Calculating pairwise distances...")
    
    # Calculate distances within real manifold
    real_to_real = pairwise_distances(real_features, real_features, metric='euclidean')
    real_radii = np.partition(real_to_real, k, axis=1)[:, k]
    
    # Calculate distances from fake to real
    fake_to_real = pairwise_distances(fake_features, real_features, metric='euclidean')
    
    # Precision: fraction of fake samples within real manifold
    precision_mask = fake_to_real.min(axis=1) <= real_radii[fake_to_real.argmin(axis=1)]
    precision = precision_mask.mean()
    
    # Recall: fraction of real samples covered by fake manifold
    fake_to_fake = pairwise_distances(fake_features, fake_features, metric='euclidean')
    fake_radii = np.partition(fake_to_fake, k, axis=1)[:, k]
    
    real_to_fake = pairwise_distances(real_features, fake_features, metric='euclidean')
    recall_mask = real_to_fake.min(axis=1) <= fake_radii[real_to_fake.argmin(axis=1)]
    recall = recall_mask.mean()
    
    return precision, recall

precision, recall = calculate_precision_recall(real_features, fake_features)

print(f"\n{'='*60}")
print(f" PRECISION: {precision:.3f}")
print(f" RECALL:    {recall:.3f}")
print(f"{'='*60}")

if precision > 0.7 and recall > 0.7:
    verdict = " EXCELLENT - Realistic AND diverse generation!"
elif precision > 0.7:
    verdict = "  HIGH PRECISION, LOW RECALL - Images look good but lack diversity"
elif recall > 0.7:
    verdict = "  HIGH RECALL, LOW PRECISION - Diverse but quality issues"
else:
    verdict = " POOR - Both quality and diversity need improvement"

print(f"\n{verdict}")

## 9️ Summary Dashboard

View all metrics together and get diagnostic recommendations.

In [ ]:
# Create metrics summary
metrics_summary = {
    'FID Score': fid_score,
    'Inception Score': is_score,
    'Precision': precision,
    'Recall': recall
}

# Create bar chart
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

metrics_names = list(metrics_summary.keys())
metrics_values = [fid_score/10, is_score, precision, recall]  # Normalize FID for visualization
colors = []

# Color code based on performance
fid_color = 'green' if fid_score < 30 else 'orange' if fid_score < 50 else 'red'
is_color = 'green' if is_score > 6 else 'orange' if is_score > 3 else 'red'
p_color = 'green' if precision > 0.7 else 'orange' if precision > 0.4 else 'red'
r_color = 'green' if recall > 0.7 else 'orange' if recall > 0.4 else 'red'
colors = [fid_color, is_color, p_color, r_color]

bars = ax.bar(metrics_names, metrics_values, color=colors, alpha=0.7, edgecolor='black', linewidth=2)

# Add value labels
for i, (bar, value, actual_value) in enumerate(zip(bars, metrics_values, [fid_score, is_score, precision, recall])):
    height = bar.get_height()
    if i == 0:  # FID (show original value)
        label = f'{actual_value:.1f}'
    else:
        label = f'{actual_value:.3f}'
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.05,
            label, ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_ylabel('Score', fontsize=13, fontweight='bold')
ax.set_title('GAN Evaluation Metrics Summary', fontsize=16, fontweight='bold')
ax.set_ylim(0, max(metrics_values) * 1.15)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print comprehensive summary
print("\n" + "="*70)
print(" "*20 + "COMPREHENSIVE EVALUATION REPORT")
print("="*70)
print(f"\nModel: {Path(MODEL_PATH).name}")
print(f"Configuration: {Path(CONFIG_PATH).name}")
print(f"\n{'-'*70}")
print(f"{'Metric':<25} {'Score':<15} {'Status':<30}")
print(f"{'-'*70}")
print(f"{'FID Score':<25} {fid_score:<15.2f} {'✅ Excellent' if fid_score < 30 else '⚠️  Needs improvement' if fid_score < 100 else '❌ Poor'}")
print(f"{'Inception Score':<25} {is_score:<15.2f} {'✅ Good' if is_score > 6 else '⚠️  Acceptable' if is_score > 3 else '❌ Poor'}")
print(f"{'Precision':<25} {precision:<15.3f} {'✅ Good' if precision > 0.7 else '⚠️  Acceptable' if precision > 0.4 else '❌ Poor'}")
print(f"{'Recall':<25} {recall:<15.3f} {'✅ Good' if recall > 0.7 else '⚠️  Acceptable' if recall > 0.4 else '❌ Poor'}")
print(f"{'-'*70}")

# Diagnostic recommendations
print(f"\n💡 DIAGNOSTIC RECOMMENDATIONS:")
if fid_score > 100:
    print(f"    Very high FID suggests mode collapse or training failure")
    print(f"      → Check if augmentation is properly configured (fill=0, not 255)")
    print(f"      → Verify learning rates are balanced (d_lr should be 2x g_lr)")
if is_score < 3:
    print(f"    Low IS indicates quality or diversity issues")
    print(f"      → Generator may not be learning all digit classes")
if precision > 0.7 and recall < 0.3:
    print(f"    High precision but low recall: generator makes realistic images but lacks variety")
    print(f"      → Try increasing augmentation or training longer")
if recall > 0.7 and precision < 0.3:
    print(f"    High recall but low precision: generator covers distribution but quality is poor")
    print(f"      → Check discriminator strength (may be too weak)")

print(f"\n" + "="*70)

## 10 Visualize Feature Space Distribution (PCA)

See how real and generated image features overlap in 2D space.

In [ ]:
# PCA visualization
print("Creating PCA visualization...")
all_features = np.vstack([real_features[:1000], fake_features[:1000]])
scaler = StandardScaler()
scaled_features = scaler.fit_transform(all_features)

pca = PCA(n_components=2)
pca_features = pca.fit_transform(scaled_features)

real_pca = pca_features[:len(real_features[:1000])]
fake_pca = pca_features[len(real_features[:1000]):]

# Plot
fig, ax = plt.subplots(1, 1, figsize=(12, 10))

ax.scatter(real_pca[:, 0], real_pca[:, 1], alpha=0.5, label='Real Images', 
           s=30, c='blue', edgecolors='none')
ax.scatter(fake_pca[:, 0], fake_pca[:, 1], alpha=0.5, label='Generated Images', 
           s=30, c='red', edgecolors='none')

# Plot means
real_mean = np.mean(real_pca, axis=0)
fake_mean = np.mean(fake_pca, axis=0)
ax.scatter(real_mean[0], real_mean[1], s=300, c='darkblue', marker='X', 
           linewidth=3, edgecolors='white', label='Real Mean', zorder=10)
ax.scatter(fake_mean[0], fake_mean[1], s=300, c='darkred', marker='X', 
           linewidth=3, edgecolors='white', label='Generated Mean', zorder=10)

# Draw line between means
ax.plot([real_mean[0], fake_mean[0]], [real_mean[1], fake_mean[1]], 
        'k--', linewidth=2, alpha=0.7, label='Distance between means')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=13, fontweight='bold')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=13, fontweight='bold')
ax.set_title('Feature Space Distribution (PCA Projection)', fontsize=16, fontweight='bold')
ax.legend(fontsize=11, loc='best')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n If blue and red clouds overlap well → Good FID score")
print(f" If clouds are separated → Poor FID score (distribution mismatch)")

---
## 📋 Summary and Recommendations

### Understanding the Metrics

| Metric | What it Measures | Good Value | Bad Value |
|--------|------------------|------------|-----------|
| **FID** | Distance between real/fake distributions | < 30 | > 100 |
| **Inception Score** | Image quality + diversity | > 6 | < 3 |
| **Precision** | Fraction of realistic generated images | > 0.7 | < 0.5 |
| **Recall** | Distribution coverage (diversity) | > 0.7 | < 0.5 |

### What to Fix Based on Results

**If FID is high (>100):**
- Check for mode collapse (all images look similar)
- Verify data augmentation isn't shifting distribution
- Consider increasing discriminator capacity
- Review learning rates (try d_lr = 2x g_lr)

**If Inception Score is low (<4):**
- Images may be blurry or unrealistic
- Try reducing learning rate
- Increase training epochs
- Check if generator is too small

**If Precision is low (<0.6):**
- Generator produces unrealistic samples
- Discriminator might be too weak
- Increase discriminator updates per generator update
- Add noise to discriminator inputs

**If Recall is low (<0.6):**
- Mode collapse detected (limited diversity)
- Reduce discriminator strength
- Add noise to labels (label smoothing)
- Increase latent dimension size

### Best Practices
- Always evaluate on **separate test set**, not training data
- Use **at least 5,000-10,000 samples** for reliable FID/IS
- Compare metrics **before and after changes** to measure improvement
- Monitor **all metrics together** - optimizing one may hurt others